### 3.1.14. Newton's Method


$$
x^{k+1}=x^k-\left[\nabla^2 f(x^k)\right]^{-1}\nabla f(x^k),
\qquad
\|x^{k+1}-x^*\| \le C\|x^k-x^*\|^2.
$$


**Explanation:**

Newton's method for optimization is obtained by replacing the cost near $x^k$ with its second-order [Taylor approximation](../../01_Foundations/03_Calculus/05_Multivariable_Differential_Calculus/10_second_order_taylor_expansion.ipynb) and minimizing that quadratic model. The related root-finding update is covered in the calculus foundation notebook on [Newton's method for root finding](../../01_Foundations/03_Calculus/02_Single_Variable_Differentiation/11_newtons_method_for_root_finding.ipynb); here the residual being driven to zero is the optimality condition $\nabla f(x)=0$.

When the Hessian is [positive definite](../../01_Foundations/01_Linear_Algebra/07_Theoretical_Linear_Algebra/17_positive_definite_matrix.ipynb), the Newton direction solves a linear system involving the Hessian and the gradient. Near a nonsingular local minimum, the model becomes increasingly accurate, the full stepsize $\alpha^k=1$ is typically accepted, and the convergence can be quadratic. The method therefore depends directly on [matrix inverse](../../01_Foundations/01_Linear_Algebra/03_Matrix/11_matrix_inverse.ipynb) ideas and on [positive definiteness](../../01_Foundations/01_Linear_Algebra/07_Theoretical_Linear_Algebra/17_positive_definite_matrix.ipynb) of the Hessian.

The important tradeoff is that Newton's method uses much richer local information than steepest descent. That information can give very fast local convergence, but it is expensive to compute and can be misleading far from the solution. These two facts must be read together: the method is powerful precisely because of its curvature model, and fragile precisely when that model is not trustworthy.

**Intuition:**

The first figure shows the rapid local behavior that motivates Newton methods. The second figure shows that pure Newton can diverge when started far from the solution, so local speed does not imply global reliability. The code cell applies the Newton minimization step to a convex objective, where each iterate solves the local quadratic model built from the objective gradient and Hessian.

<img src="../../Figures/030101_berts_fig_1_4_1_fast_newton_convergence.jpeg" alt="Bertsekas Figure 1.4.1: fast local convergence of Newton method" style="width: 60%; height: auto;">

<img src="../../Figures/030101_berts_fig_1_4_2_newton_divergence.jpeg" alt="Bertsekas Figure 1.4.2: Newton method can diverge from a poor starting point" style="width: 60%; height: auto;">


**Numerical Example:**

Minimize the convex objective
$$
f(x)=e^x-x.
$$

The gradient and Hessian are
$$
f'(x)=e^x-1,
\qquad
f''(x)=e^x>0.
$$

Newton's minimization update is therefore
$$
x^{k+1}=x^k-\frac{f'(x^k)}{f''(x^k)}
=x^k-\frac{e^{x^k}-1}{e^{x^k}}.
$$

Start at $x^0=1$. Since $e^1\approx 2.7183$,
$$
x^1=1-\frac{2.7183-1}{2.7183}
=1-0.6321
=0.3679.
$$

At the next step, $e^{0.3679}\approx 1.4447$, so
$$
x^2=0.3679-\frac{1.4447-1}{1.4447}
=0.3679-0.3078
=0.0601.
$$

The optimality condition $f'(x^*)=0$ gives $x^*=0$, and the iterates move rapidly toward that minimizer once the local quadratic model is accurate.


In [1]:
import sympy as sp

variable = sp.symbols("x")
objective = sp.exp(variable) - variable
gradient = sp.diff(objective, variable)
hessian = sp.diff(gradient, variable)
current_point = sp.Float(1.0)
newton_points = []

for iteration in range(5):
    gradient_value = gradient.subs(variable, current_point)
    hessian_value = hessian.subs(variable, current_point)
    current_point = current_point - gradient_value / hessian_value
    newton_points.append(float(current_point))

print("objective =", objective)
print("gradient =", gradient)
print("hessian =", hessian)
print("newton_points =", newton_points)


objective = -x + exp(x)
gradient = exp(x) - 1
hessian = exp(x)
newton_points = [0.36787944117144233, 0.06008006872678873, 0.00176919944264467, 1.5641107898984284e-06, 1.2233215659894107e-12]


**Equivalent `casadi` implementation:**

In [2]:
import casadi as ca

variable = ca.SX.sym("x")
objective = ca.exp(variable) - variable
gradient = ca.gradient(objective, variable)
hessian = ca.hessian(objective, variable)[0]
gradient_function = ca.Function("grad_f", [variable], [gradient])
hessian_function = ca.Function("hess_f", [variable], [hessian])

current_point = ca.DM(1.0)
newton_points = []
for iteration in range(5):
    current_point = current_point - gradient_function(current_point) / hessian_function(current_point)
    newton_points.append(float(current_point))

print("newton_points =", newton_points)


newton_points = [0.36787944117144233, 0.06008006872678873, 0.00176919944264467, 1.5641107898984284e-06, 1.2233215659894107e-12]


**References:**

[📘 Bertsekas, D. P. (1999). *Nonlinear Programming* (2nd ed.), Chapter 1 "Unconstrained Optimization". Athena Scientific.](https://www.athenasc.com/nonlinbook.html)

---

[⬅️ Previous: Scaling and Preconditioning](./13_scaling_and_preconditioning.ipynb) | [Next: Damped and Modified Newton Methods ➡️](./15_damped_and_modified_newton_methods.ipynb)

---
